# pMF Any-to-Any no Colab T4

Liga um runtime GPU T4. O notebook obtém o código por Git e reutiliza o export
RF100 Roboflow v1. Se já fizeste o smoke com sucesso neste runtime, salta para
**Treino completo**. As células de monitorização podem ser corridas após cada época.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO = Path("/content/Synthetic-Data-Generation-Framework")
DATASET = Path("/content/rf100-bi3q3")
CACHE = Path("/content/pmf_any_to_any_features")
RUN = Path("/content/pmf_any_to_any_run")
BACKBONE = None
REPO_URL = "https://github.com/andreferraria453/Synthetic-Data-Generation-Framework.git"

subprocess.run(["nvidia-smi", "-L"], check=True)
if not REPO.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(["git", "pull", "--ff-only", "origin", "main"], cwd=REPO, check=True)

def run(*args):
    subprocess.run([sys.executable, *map(str, args)], cwd=REPO, check=True)

if not (DATASET / "data.yaml").is_file():
    try:
        from huggingface_hub import snapshot_download
    except ModuleNotFoundError:
        run("-m", "pip", "install", "huggingface_hub")
        from huggingface_hub import snapshot_download

    snapshot_download(
        repo_id="LibreYOLO/corrosion-bi3q3",
        repo_type="dataset",
        revision="d20b5b1e6b21ff7de555bbf848dcf7d30ccf8ee7",
        local_dir=str(DATASET),
    )

if not (REPO / "configs/pmf_any_to_any.json").is_file():
    raise FileNotFoundError(f"Falta configs/pmf_any_to_any.json em {REPO}")
if not (DATASET / "data.yaml").is_file():
    raise FileNotFoundError(f"Falta data.yaml em {DATASET}")

def experiment_args():
    args = [
        "-m", "rae_experiment.any_to_any.train",
        "--config", "configs/pmf_any_to_any.json",
        "--dataset", DATASET, "--cache", CACHE, "--output", RUN,
    ]
    if BACKBONE is not None:
        args.extend(["--backbone", BACKBONE])
    return args

print(f"Código: {REPO} | Dataset: {DATASET}")

## Dependências

Executa a instalação uma vez por runtime e reinicia o kernel se o Colab o pedir. Mantém o PyTorch CUDA do runtime. A T4 usa FP32 neste preflight.

In [6]:
run("-m", "pip", "install", "-r", "requirements-any-to-any.txt")

## Auditoria, precompute curto e smoke test

O precompute usa duas imagens do treino e duas da validação. A primeira execução pode carregar checkpoints dos encoders e do pMF para a cache do Colab.

In [7]:
run("-m", "unittest", "discover", "-s", "tests", "-v")
run("-m", "rae_experiment.any_to_any.data", "--dataset", DATASET, "--audit-only")
run("-m", "rae_experiment.any_to_any.data", "--dataset", DATASET,
    "--cache", CACHE, "--limit", "2")

In [ ]:
run(*experiment_args(), "--smoke")

## Treino completo

Se já executaste o smoke neste runtime, podes correr só esta célula com as
variáveis de configuração existentes. Indica o checkpoint do detetor RF100,
confirma o relatório do smoke e muda START_TRAINING para True. O precompute
aproveita a cache da amostra. As 50 épocas têm validação por época e checkpoint
a cada cinco épocas. Para retomar, define RESUME para o checkpoint pretendido.

In [ ]:
START_TRAINING = False  # Muda para True depois do smoke passar.
DETECTOR = Path("/content/best.pt")  # Checkpoint Ultralytics RF100 treinado em train.
RESUME = None  # Ou RUN / "checkpoints" / "epoch-005.pt"

if START_TRAINING:
    if not (RUN / "smoke.json").is_file():
        raise FileNotFoundError(f"Falta o relatório do smoke em {RUN / 'smoke.json'}")
    if not DETECTOR.is_file():
        raise FileNotFoundError(f"Indica o checkpoint do detetor RF100 em DETECTOR: {DETECTOR}")
    if RESUME is not None and not Path(RESUME).is_file():
        raise FileNotFoundError(RESUME)

    run("-m", "rae_experiment.any_to_any.data", "--dataset", DATASET, "--cache", CACHE)

    args = [
        "-m", "rae_experiment.any_to_any.train",
        "--config", "configs/pmf_any_to_any.json",
        "--dataset", DATASET, "--cache", CACHE, "--output", RUN,
        "--detector", DETECTOR, "--train",
    ]
    if BACKBONE is not None:
        args.extend(["--backbone", BACKBONE])
    if RESUME is not None:
        args.extend(["--resume", RESUME])
    run(*args)
else:
    print("Treino em pausa. Define START_TRAINING = True para arrancar.")

## Curvas e amostras

Define RUN com o diretório da experiência e volta a correr estas células à medida que forem sendo escritas épocas.

In [9]:
import json
import matplotlib.pyplot as plt
from IPython.display import Image, display

rows = [json.loads(path.read_text()) for path in sorted((RUN / "epochs").glob("*.json"))]
if not rows:
    print("Ainda não existem épocas completas.")
for group in ("train", "validation", "metrics", "efficiency") if rows else ():
    series = [row[group] if group in row else {
        key: val for key, val in row.items() if key.endswith(("_seconds", "_mib"))
    } for row in rows]
    keys = sorted({key for values in series for key in values})
    for offset in range(0, len(keys), 4):
        figure, axes = plt.subplots(1, 4, figsize=(16, 3), constrained_layout=True)
        for axis in axes:
            axis.set_visible(False)
        for axis, key in zip(axes, keys[offset:offset + 4]):
            axis.set_visible(True)
            values = [(row["epoch"], entry[key]) for row, entry in zip(rows, series) if entry.get(key) is not None]
            if values:
                axis.plot(*zip(*values), marker=".")
            axis.set_title(f"{group}: {key}")
            axis.set_xlabel("Época")
        plt.show()

Ainda não existem épocas completas.


In [10]:
EPOCH = None  # Inteiro para selecionar uma época específica
examples = sorted((RUN / "examples").glob("*/rgb_reference_generated_edit.png"))
if EPOCH is not None:
    examples = [path for path in examples if path.parent.name == f"{EPOCH:03}"]
if examples:
    print("Referência · geração livre · edição localizada")
    display(Image(filename=str(examples[-1])))